# Exercice 10 : Enrichir par une jointure

À partir du fichier nettoyé à l'exercice 08.

In [8]:
import pandas as pd

titanic = pd.read_csv("../exercice-08/data/titanic_propre.csv")
print(titanic.shape)

(889, 9)


## 1. La table des ports

In [9]:
ports = pd.DataFrame({
    "embark_town": ["Southampton", "Cherbourg", "Queenstown"],
    "pays": ["Angleterre", "France", "Irlande"],
})
print(ports)

   embark_town        pays
0  Southampton  Angleterre
1    Cherbourg      France
2   Queenstown     Irlande


## 2. Jointure en `left` et contrôle du nombre de lignes

In [10]:
nb_avant = len(titanic)
titanic_pays = titanic.merge(ports, on="embark_town", how="left",
                             validate="many_to_one")
assert len(titanic_pays) == nb_avant
print(f"Lignes avant : {nb_avant}, après : {len(titanic_pays)}")

Lignes avant : 889, après : 889


## 3. Passagers et taux de survie par pays

In [11]:
survie_par_pays = titanic_pays.groupby("pays").agg(
    passagers=("survived", "size"),
    taux_survie=("survived", "mean"),
).round(2)
print(survie_par_pays)

            passagers  taux_survie
pays                              
Angleterre        644         0.34
France            168         0.55
Irlande            77         0.39


Les passagers embarqués en France (Cherbourg) ont le mieux survécu : 55 %, contre 39 % pour l'Irlande et 34 % pour l'Angleterre.

## 4. Sans Queenstown : jointure `inner` puis `left`

In [12]:
ports_sans_queenstown = ports[ports["embark_town"] != "Queenstown"]
print(ports_sans_queenstown)

   embark_town        pays
0  Southampton  Angleterre
1    Cherbourg      France


In [13]:
jointure_inner = titanic.merge(ports_sans_queenstown, on="embark_town", how="inner",
                               validate="many_to_one")
print(f"inner : {len(jointure_inner)} lignes sur {nb_avant}")
print(jointure_inner["pays"].value_counts(dropna=False))

inner : 812 lignes sur 889
pays
Angleterre    644
France        168
Name: count, dtype: int64


In [14]:
jointure_left = titanic.merge(ports_sans_queenstown, on="embark_town", how="left",
                              validate="many_to_one")
print(f"left : {len(jointure_left)} lignes sur {nb_avant}")
print(jointure_left["pays"].value_counts(dropna=False))

left : 889 lignes sur 889
pays
Angleterre    644
France        168
NaN            77
Name: count, dtype: int64


- **En `inner`**, on passe de 889 à 812 lignes : les 77 passagers embarqués à Queenstown disparaissent sans message d'erreur, car leur port n'a plus de correspondance dans `ports`.
- **En `left`**, on garde bien les 889 lignes, mais la colonne `pays` est vide (`NaN`) pour ces 77 passagers.

Aucune des deux jointures ne prévient : en `inner` on perd des passagers, en `left` on crée des manquants. D'où l'intérêt de compter les lignes avant et après, et de recompter les manquants après une jointure.